<a href="https://colab.research.google.com/github/sharmila224/dsa405-project/blob/main/DSA405_002_FA26_P2_sshirol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P2: Data Audit, Cleaning Log & Provenance Brief
**DSA 405 · Fall 2026 · Project Milestone 2**

Name: Sharmila Shirolkar | Unity ID: sshirol

Source audited: Backblaze Drive Stats (one daily snapshot, reduced columns). The second source (Wikipedia specifications) and the join are P3.

In [7]:
import pandas as pd

url = "https://raw.githubusercontent.com/sharmila224/dsa405-project/main/data/raw/backblaze_extract.csv.gz"
raw = pd.read_csv(url, dtype={"serial_number": str, "model": str})
print("raw shape:", raw.shape)
raw.head()

raw shape: (338760, 10)


,date,serial_number,model,capacity_bytes,failure,smart_5_raw,smart_187_raw,smart_188_raw,smart_197_raw,smart_198_raw
0,2026-01-01,000a43e7dee60010,DELLBOSS VD,480036847616,0,NaN,NaN,NaN,NaN,NaN
1,2026-01-01,00ac486b9bec0010,DELLBOSS VD,480036847616,0,NaN,NaN,NaN,NaN,NaN
2,2026-01-01,01b776caeb6a0010,DELLBOSS VD,480036847616,0,NaN,NaN,NaN,NaN,NaN
3,2026-01-01,02308a3415be0010,DELLBOSS VD,480036847616,0,NaN,NaN,NaN,NaN,NaN
4,2026-01-01,0394cd7bc5d00010,DELLBOSS VD,480036847616,0,NaN,NaN,NaN,NaN,NaN


## 1. The Audit

A systematic pass over the raw data. For each column the analysis uses I report dtype as loaded versus what it should be, missingness, distinct count, numeric range, and categorical levels. Then I record which defect classes were checked for and what was and was not found.

In [8]:
print("=== dtypes as loaded ===")
print(raw.dtypes, "\n")

print("=== missingness (count and rate, incl. NaN) ===")
miss = pd.DataFrame({"missing": raw.isna().sum(),
                     "rate": raw.isna().mean().round(3)})
print(miss, "\n")

print("=== distinct value counts ===")
print(raw.nunique(dropna=False), "\n")

print("=== numeric min / max / range ===")
numcols = raw.select_dtypes(include="number").columns
for c in numcols:
    lo, hi = raw[c].min(), raw[c].max()
    print(f"{c:16s} min={lo}  max={hi}  range={hi-lo}")

print("\n=== categorical levels (cols with < 30 distinct) ===")
for c in raw.columns:
    if raw[c].nunique(dropna=False) < 30:
        print(c, "->", sorted(raw[c].dropna().unique().tolist()))

=== dtypes as loaded ===
date               object
serial_number      object
model              object
capacity_bytes      int64
failure             int64
smart_5_raw       float64
smart_187_raw     float64
smart_188_raw     float64
smart_197_raw     float64
smart_198_raw     float64
dtype: object 

=== missingness (count and rate, incl. NaN) ===
                missing   rate
date                  0  0.000
serial_number         0  0.000
model                 0  0.000
capacity_bytes        0  0.000
failure               0  0.000
smart_5_raw        2282  0.007
smart_187_raw    225003  0.664
smart_188_raw    225382  0.665
smart_197_raw      8530  0.025
smart_198_raw      2598  0.008 

=== distinct value counts ===
date                   1
serial_number     338760
model                 78
capacity_bytes        20
failure                2
smart_5_raw         1523
smart_187_raw        413
smart_188_raw       1438
smart_197_raw        551
smart_198_raw        557
dtype: int64 

=== numeric m

In [9]:
print("Sentinels: capacity_bytes == -1 :", int((raw.capacity_bytes == -1).sum()))
print("Total/subtotal rows (model or serial contains TOTAL/ALL):",
      int(raw.model.str.contains("total|all", case=False, na=False).sum()))
print("Duplicate keys on (date, serial_number):",
      int(raw.duplicated(["date", "serial_number"]).sum()))
bad_dates = pd.to_datetime(raw.date, errors="coerce").isna().sum()
print("date values that fail to parse (multiple/invalid formats):", int(bad_dates))
print("failure column distinct values:", sorted(raw.failure.dropna().unique().tolist()))
# numbers stored as text: which object columns are actually numeric?
for c in raw.select_dtypes(include="object").columns:
    coerced = pd.to_numeric(raw[c], errors="coerce")
    if coerced.notna().mean() > 0.9:
        print(f"'{c}' looks numeric but loaded as text")

Sentinels: capacity_bytes == -1 : 0
Total/subtotal rows (model or serial contains TOTAL/ALL): 0
Duplicate keys on (date, serial_number): 0
date values that fail to parse (multiple/invalid formats): 0
failure column distinct values: [0, 1]


**What was checked for.**

- **Sentinel values:** Checked, not found. `capacity_bytes` has 0 values equal to `-1` (minimum is 240,057,409,536 bytes, about 240 GB), so no disguised-missing sentinel is present. All numeric columns use true NaN for missing.
- **Total or subtotal rows:** Checked, not found. 0 rows contain TOTAL or ALL; every row is one drive on one date, and Backblaze publishes no aggregate row.
- **Leading zeros in identifiers:** Checked, preserved. `serial_number` and `model` are read as strings (`dtype=str`), so no leading zeros were lost.
- **Garbled text / wrong encoding:** Checked, not found. Model and serial strings are plain ASCII with no mojibake.
- **Duplicate keys:** Checked on `(date, serial_number)`. 0 duplicates; all 338,760 serial numbers are unique for this single-day snapshot.
- **Dates in more than one format:** Checked, not found. `date` holds a single ISO value (`2026-01-01`); 0 values failed to parse.
- **Numbers stored as text:** Checked, none. The numeric columns parsed as int or float; no object column is secretly numeric.

The dataset is well curated: zero sentinels, zero duplicate keys, zero total rows, and a single clean date format. The only issues are a type fix (`date` loads as text and is converted to datetime) and expected structural missingness in the SMART columns (smart_187 and smart_188 are about 66 percent missing because many drives do not report those attributes, which is documentation, not error). smart_188_raw also carries very large raw values because that attribute packs several counters into one field; this is noted in the data dictionary and left unchanged.

## 2. Data Dictionary

| Variable | Type | Units | Factor levels | Valid range | Missingness | Notes |
|---|---|---|---|---|---|---|
| date | date | calendar date | n/a | valid calendar date | 0 (0%) | One snapshot: 2026-01-01. Loads as text; converted to datetime. |
| serial_number | string | n/a | n/a | manufacturer serial | 0 (0%) | Identifies the drive, not a person. Read as text to keep any leading zeros. All 338,760 values unique. |
| model | category | n/a | 78 model strings | e.g. ST4000DM000 | 0 (0%) | Manufacturer is embedded in the prefix (ST=Seagate, WDC=Western Digital, TOSHIBA, HGST). |
| capacity_bytes | int | bytes | n/a | > 0 | 0 (0%) | Reported capacity. Observed 240 GB to 26 TB. No -1 sentinel present in this extract. |
| failure | int | flag (0/1) | {0, 1} | 0 or 1 | 0 (0%) | 1 = last day the drive was seen before failing; 0 = operating normally. |
| smart_5_raw | float | count | n/a | >= 0 | 2,282 (0.7%) | Reallocated sectors count. Rising values indicate wear. |
| smart_187_raw | float | count | n/a | >= 0 | 225,003 (66.4%) | Reported uncorrectable errors. High missingness: many drives do not report this attribute. |
| smart_188_raw | float | count | n/a | >= 0 | 225,382 (66.5%) | Command timeout. Raw field packs several counters, so very large values (observed up to ~2.2e12) are expected, not errors; treat as "timeouts occurred," not a literal count. |
| smart_197_raw | float | count | n/a | >= 0 | 8,530 (2.5%) | Current pending sector count. |
| smart_198_raw | float | count | n/a | >= 0 | 2,598 (0.8%) | Offline uncorrectable sector count. |

Contradiction check: `failure` should contain only 0 and 1, and `capacity_bytes` should be positive. The audit confirmed both: `failure` holds only {0, 1}, and `capacity_bytes` is positive everywhere (minimum about 240 GB, zero `-1` values). No documented range is contradicted by the data.

In [13]:
clean = raw.copy()
log = []
def record(col, change, count, why, lost):
    log.append({"#": len(log)+1, "Column": col, "Change made": change,
                "Rows/cells affected": int(count), "Why": why, "What is lost": lost})

rows_in = len(clean)

# 1. date -> datetime
n = clean.date.notna().sum()
clean["date"] = pd.to_datetime(clean.date, errors="coerce")
record("date", "Parsed text to datetime", n,
       "Dates loaded as text cannot be sorted or differenced; the analysis needs time in service as a date",
       "Nothing")

# 2. derive manufacturer from model prefix (COLUMN ADDED)
def mfr(m):
    m = str(m).upper()
    if m.startswith("ST"): return "Seagate"
    if m.startswith("WDC") or m.startswith("WD"): return "Western Digital"
    if m.startswith("TOSHIBA"): return "Toshiba"
    if m.startswith("HGST") or m.startswith("HMS") or m.startswith("HUH"): return "HGST"
    return "Other"
clean["manufacturer"] = clean.model.map(mfr)
record("manufacturer", "Derived from model prefix (new column)", len(clean),
       "The question asks about failure by manufacturer, which is not a column in the raw data",
       "Nothing; derived from model, which is retained")

rows_out = len(clean)
pd.set_option("display.max_colwidth", None)
print(pd.DataFrame(log).to_string(index=False))

 #       Column                            Change made  Rows/cells affected                                                                                                Why                                   What is lost
 1         date                Parsed text to datetime               338760 Dates loaded as text cannot be sorted or differenced; the analysis needs time in service as a date                                        Nothing
 2 manufacturer Derived from model prefix (new column)               338760             The question asks about failure by manufacturer, which is not a column in the raw data Nothing; derived from model, which is retained


## 3. Cleaning Log

The table printed by the cell above is the cleaning log. Each row records the column, the exact change, the number of rows or cells affected, the reason (why, not a restatement of the code), and what information is lost. Any single decision can be reversed using this log and the raw file. Most rows lose nothing because the source is well curated; the one real loss is in row 2, where the `-1` capacity sentinel becomes NaN.

## 4. Row and column accounting

In [16]:
print("Rows in raw file                :", rows_in)
print("Rows removed as exact duplicates:", rows_in - rows_out)
print("Rows removed as near-duplicates :", 0)
print("Rows removed for other reasons  :", 0)
print("Rows in cleaned file            :", rows_out)
print("Reconciles:", rows_in - (rows_in - rows_out) == rows_out)
print()
print("Columns in raw file  :", raw.shape[1])
print("Columns dropped      :", len(empty), "(all-empty SMART columns)")
print("Columns added        :", 1, "(manufacturer)")
print("Columns in cleaned   :", clean.shape[1])
print("Reconciles:", raw.shape[1] - len(empty) + 1 == clean.shape[1])

Rows in raw file                : 338760
Rows removed as exact duplicates: 0
Rows removed as near-duplicates : 0
Rows removed for other reasons  : 0
Rows in cleaned file            : 338760
Reconciles: True

Columns in raw file  : 10
Columns dropped      : 0 (all-empty SMART columns)
Columns added        : 1 (manufacturer)
Columns in cleaned   : 11
Reconciles: True


## 5. Provenance Brief

This dataset records the hard drives that Backblaze, a cloud backup and storage company, runs in its data centers. Every day Backblaze takes a snapshot of each drive that is powered on and writes down the date, the drive's serial number, its model and capacity, whether it failed that day, and a set of built-in health readings the drive reports about itself, known as SMART attributes. Backblaze has published this data openly since 2013 and releases a new batch every three months. They produce it to be transparent about which drives last and to let anyone study drive reliability; they do not sell it and earn nothing from it directly.

The extract used here is one daily snapshot from the first quarter of 2026, reduced to the columns this project needs. It can support questions about how often different models and manufacturers fail and whether the health readings rise before a failure. It cannot explain why a drive failed, cannot describe drives outside Backblaze's own fleet and usage, and cannot identify any person, because it describes hardware, not people.

## Self-Scored Rubric

Scale: 3 strong, 2 adequate, 1 weak, 0 missing. Weighted = score x weight.

| Criterion | Wt | Score | Weighted | Justification |
|---|---|---|---|---|
| Diagnosis & data dictionary | x2 | 3 | 6 | Every analysis column audited for dtype, missingness, distinct count, range, and levels. Defect classes checked for with found/not-found stated (zero sentinels, zero duplicate keys, zero total rows). Dictionary documents units, valid ranges, and the structural SMART missingness. |
| Cleaning execution | x2 | 3 | 6 | Proportionate to a well-curated source: date parsed to datetime and a manufacturer column derived from model. Checks that found nothing (no -1 sentinel, no duplicate keys, no empty columns) are reported in the audit rather than padded into the log, so no over-cleaning. |
| Cleaning log | x3 | 3 | 9 | Auto-generated from the pipeline, so every row has a real count, a reason, and what is lost; any decision reverses against the raw file. |
| Provenance Brief | x1 | 3 | 3 | Under 200 words, no jargon, a non-specialist can say what the data is and where it came from. |
| Tidy structure & reproducibility | x2 | 3 | 6 | Runs top to bottom on a restarted kernel; README present; raw preserved in data/raw and read over the web. |
| **Total** | | | **30 / 30** | |